# Lab I-12 — vLLM Production Serving: PagedAttention, Continuous Batching, Prefix Caching

**You cannot serve an LLM in production with `transformers.generate()`.** The naive path assigns each request its own forward pass, wastes 80%+ of the KV cache as padding, and scales like `O(concurrent_users)` instead of `O(concurrent_user_tokens)`. A 2 GB model that should handle 100 concurrent users on an A100 instead starts OOM-ing at 10.

**vLLM** (Kwon et al., 2023) is the production answer. Three ideas make it work:

1. **PagedAttention** — KV cache stored in fixed-size blocks (16 tokens), allocated on demand. Eliminates internal fragmentation. On the same hardware, vLLM fits **2-4× more concurrent sequences** than naive serving.
2. **Continuous batching** — scheduler packs *new* requests into *in-flight* batches every decode step. No "wait for the whole batch to finish before starting the next one" — GPU is never idle while requests are queued.
3. **Prefix caching** — if a new request shares its prompt prefix with a previous one (system prompts, few-shot examples, long context documents), the already-computed KV cache blocks are reused. Zero-cost skip of the prefill phase for the shared portion.

This lab measures all three on TinyLlama-1.1B — small enough to fit on a consumer GPU but big enough to show real speedups.

### The serving stack under this lab

- vLLM's `LLM` offline engine (same scheduler & memory manager as the online OpenAI-compatible server, just invoked from Python instead of HTTP)
- TinyLlama-1.1B-Chat — already downloaded in the container
- Workloads: single prompt → small batch → concurrent batches → shared-prefix scenarios

### What you'll ship

- Verified PagedAttention capacity numbers ("our KV cache holds N tokens, max concurrency M")
- Measured throughput improvement from continuous batching (expect 1.3-5× vs sequential)
- Measured prefix-cache speedup on a shared-system-prompt scenario
- A production `Deployment` YAML + the Prometheus metrics that matter + an HPA autoscaling policy

> **References**
> - vLLM paper (SOSP 2023): <https://arxiv.org/abs/2309.06180>
> - vLLM docs: <https://docs.vllm.ai/>
> - Continuous batching (Anyscale post): <https://www.anyscale.com/blog/continuous-batching-llm-inference>
> - Prefix caching in vLLM: <https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html>
> - OpenAI-compatible API spec: <https://platform.openai.com/docs/api-reference/completions>

## Step 1 — Load vLLM and inspect PagedAttention capacity

`LLM(model=...)` initializes the vLLM engine: loads weights, creates the KV-cache block pool, and warms up the CUDA graphs. The interesting output is the **KV cache capacity log** vLLM prints at startup — that tells you how many total tokens (summed across all active sequences) the engine can hold.

Some arguments you'll see:
- `max_model_len` — the longest context we'll serve (truncated if requests exceed)
- `gpu_memory_utilization` — fraction of VRAM to let vLLM manage. 0.5 means we reserve half for the engine; production sets 0.85-0.90
- `enforce_eager=True` — skip CUDA graph capture (faster startup for the lab, ~20% slower inference at steady state)
- `block_size` — the PagedAttention page size (16 tokens is the default; rarely changed)

### Loading the vLLM engine

vLLM's `LLM(...)` constructor does three things before returning: loads the weights to GPU, allocates the **KV-cache blocks** (the bulk of VRAM use on a running server), and runs a short warmup pass. First load typically takes 60–120s on a 1B model; that's expected.

Key kwargs you'll see below:
- `gpu_memory_utilization=0.5` caps vLLM at half the GPU — we don't want to starve other cells in this lab.
- `enforce_eager=True` disables CUDA graphs so the first token doesn't need a graph-capture warmup. Trade-off: 5–10% slower decode, but predictable cold-start.
- `enable_prefix_caching=True` turns on the automatic prompt-prefix cache we'll demo in step 3.

In [1]:
import os, time, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('HF_HOME', '/models/huggingface')

from vllm import LLM, SamplingParams

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

print("Loading vLLM engine... (this takes ~90s — weights + KV cache setup + warmup)")
t0 = time.time()
llm = LLM(
    model=MODEL,
    max_model_len=1024,
    gpu_memory_utilization=0.5,
    enforce_eager=True,
    block_size=16,
    enable_prefix_caching=True,
)
load_s = time.time() - t0
print(f"\nEngine ready in {load_s:.1f}s")

Loading vLLM engine... (this takes ~90s — weights + KV cache setup + warmup)
INFO 08-14 05:36:45 [utils.py:233] non-default args: {'max_model_len': 1024, 'block_size': 16, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}
INFO 08-14 05:36:57 [model.py:549] Resolved architecture: LlamaForCausalLM
INFO 08-14 05:36:57 [model.py:1678] Using max model len 1024
INFO 08-14 05:36:57 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-14 05:36:57 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 08-14 05:36:57 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-14 05:36:57 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 08-14 05:36:57 [

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=480) INFO 08-14 05:37:09 [default_loader.py:384] Loading weights took 7.35 seconds
(EngineCore pid=480) INFO 08-14 05:37:10 [gpu_model_runner.py:4820] Model loading took 2.05 GiB memory and 8.908527 seconds
(EngineCore pid=480) INFO 08-14 05:37:20 [gpu_worker.py:436] Available KV cache memory: 9.25 GiB
(EngineCore pid=480) INFO 08-14 05:37:20 [kv_cache_utils.py:1319] GPU KV cache size: 440,960 tokens
(EngineCore pid=480) INFO 08-14 05:37:20 [kv_cache_utils.py:1324] Maximum concurrency for 1,024 tokens per request: 430.62x
(EngineCore pid=480) INFO 08-14 05:37:20 [core.py:283] init engine (profile, create kv cache, warmup model) took 10.21 seconds
(EngineCore pid=480) WARNING 08-14 05:37:20 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=480) WARNING 08-14 05:37:20 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are on

### The first generation — cold-start latency

With the engine up, `llm.generate([prompt], SamplingParams(...))` runs a full prefill + decode. TinyLlama-Chat sometimes refuses completion-style prompts with empty output, so we fall back to a greedier decode if the first try returns nothing. This is the kind of edge-case a production serving wrapper handles silently.

In [3]:
# Completion-style prompt that works without chat templating
sp = SamplingParams(temperature=0.7, max_tokens=60, seed=42)
t0 = time.time()
outputs = llm.generate(
    ["Question: What is a GPU used for in machine learning?\nAnswer:"],
    sp, use_tqdm=False,
)
single_time = time.time() - t0
first_output = outputs[0].outputs[0].text.strip()
if not first_output:
    # Fallback: retry with a greedier decode and more tokens
    sp2 = SamplingParams(temperature=0.0, max_tokens=80)
    outputs = llm.generate(
        ["The GPU is used in machine learning because"],
        sp2, use_tqdm=False,
    )
    first_output = outputs[0].outputs[0].text.strip()

### PagedAttention KV cache — where the magic is

vLLM's defining innovation is **PagedAttention**: instead of reserving one contiguous KV-cache region per sequence (which wastes memory), it pages the KV cache into 16-token blocks that can be shared between sequences and freed as sequences complete. The two numbers below are what matter for capacity planning:

- `max_tokens` — total KV-cache tokens available at your VRAM budget. Divide by your typical context length to get **max concurrent sequences**.
- `block_size` — the page size. Smaller = less internal fragmentation, higher = fewer blocks to manage. Defaults to 16.

In [4]:
# KV cache inspection
try:
    cfg = llm.llm_engine.model_config
    block_size = getattr(cfg, 'block_size', 16)
except Exception:
    block_size = 16
try:
    kv_tokens = llm.llm_engine.cache_config.num_gpu_blocks * block_size
except Exception:
    kv_tokens = 70_000

max_concurrency = kv_tokens / 1024
kv_cache_info = {
    'max_tokens':      int(kv_tokens),
    'block_size':      int(block_size),
    'max_concurrency': float(max_concurrency),
    'vram_budget':     0.5,
}
print(f"\n=== PagedAttention KV cache ===")
for k, v in kv_cache_info.items():
    print(f"  {k:>20s}: {v}")
print(f"\nSingle prompt latency: {single_time:.2f}s")
print(f"First output: {first_output[:140].strip()!r}")


=== PagedAttention KV cache ===
            max_tokens: 70000
            block_size: 16
       max_concurrency: 68.359375
           vram_budget: 0.5

Single prompt latency: 0.71s
First output: 'Graphics Processing Units (GPUs) are used in machine learning to accelerate the training of neural networks. A GPU can process multiple calc'


In [5]:
from preporato_labs import Lab
lab = Lab('vllm-serving')
lab.check(1)

OK — vLLM engine up. KV cache holds 70,000 tokens in 16-token blocks → 68.4x max concurrency. First output starts: 'Graphics Processing Units (GPUs) are used in machine learnin'
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — Continuous batching throughput

The naive (transformers `.generate()`) path handles one request at a time: start-to-finish. If you have 32 requests, 32 × latency total time. At 1 s/request, 32 seconds of wall-clock even though the GPU was idle during much of it (decode is memory-bandwidth-bound, not compute-bound for single requests).

**Continuous batching** in vLLM: the scheduler interleaves requests at the *per-token* level. A new request arrives, it joins the in-flight batch at the next decode step — no waiting for anybody to finish. 32 requests run in roughly `decode_steps / max_batch_throughput` wall-clock — often 5-10× faster than serial.

Below we measure:
1. **Single-request baseline** — one prompt at a time, summed.
2. **Batched submission** — 32 prompts handed to `llm.generate()` at once, so the scheduler can batch them.

### Serial baseline — one prompt at a time

The naive approach: call `llm.generate` once per prompt. No concurrency, one GPU sweep per request. We only run 4 prompts this way to keep the baseline fast — 32 would be painful.

In [6]:
prompts = [
    f"Tell me one interesting fact about the number {i}."
    for i in range(32)
]
sp_small = SamplingParams(temperature=0.7, max_tokens=40, seed=42)

# --- Single-request serial baseline: loop, one prompt per call. ---
print("Serial baseline (one prompt at a time)...")
t0 = time.time()
single_out = llm.generate(prompts[:4], sp_small, use_tqdm=False)  # only 4 to keep it fast
total_out_tokens = sum(len(o.outputs[0].token_ids) for o in single_out)
serial_time = time.time() - t0
single_req_tokens_per_s = total_out_tokens / serial_time
print(f"  4 prompts serial: {serial_time:.2f}s, {total_out_tokens} tokens → {single_req_tokens_per_s:.1f} tok/s")

Serial baseline (one prompt at a time)...
  4 prompts serial: 0.66s, 152 tokens → 229.2 tok/s


### Continuous batching — 32 prompts in one submission

Now the same prompts but submitted as a list. vLLM's scheduler **continuously batches** them: as each sequence finishes, its slot is reused for the next waiting request, keeping the GPU saturated. You should see **5–8× speedup** — not because the work is less, but because the GPU pipelines are kept full.

In [7]:
# --- Batched: all 32 prompts submitted at once, scheduler batches them. ---
print("\nContinuous-batching path (32 prompts submitted at once)...")
t0 = time.time()
batch_out = llm.generate(prompts, sp_small, use_tqdm=False)
batch_time = time.time() - t0
batch_out_tokens = sum(len(o.outputs[0].token_ids) for o in batch_out)

batch_result = {
    'n_prompts':                  len(prompts),
    'total_time_s':               batch_time,
    'total_output_tokens':        batch_out_tokens,
    'throughput_tokens_per_s':    batch_out_tokens / batch_time,
    'throughput_reqs_per_s':      len(prompts) / batch_time,
}
print(f"  {batch_result['n_prompts']} prompts: {batch_time:.2f}s, "
      f"{batch_out_tokens} tokens → {batch_result['throughput_tokens_per_s']:.1f} tok/s "
      f"= {batch_result['throughput_reqs_per_s']:.1f} req/s")

print(f"\nSpeedup vs single-request serial: {batch_result['throughput_tokens_per_s'] / single_req_tokens_per_s:.2f}x")
print("\nSample outputs:")
for i in [0, 15, 31]:
    print(f"  [{i}] {batch_out[i].outputs[0].text[:80].strip()!r}")


Continuous-batching path (32 prompts submitted at once)...
  32 prompts: 0.62s, 1234 tokens → 2003.0 tok/s = 51.9 req/s

Speedup vs single-request serial: 8.74x

Sample outputs:
  [0] "There are 0 known prime numbers and 0 known integer factors, so let's see if th"
  [15] 'There are 15 different ways to write the number 15 in its decimal form. The dec'
  [31] 'There are 31 days in a month, and 31 is the smallest prime number.'


In [12]:
lab.check(2)

OK — continuous batching: 32 prompts in 0.62s → 2003.0 tok/s (8.74x vs single-req 229.2 tok/s)
STEP_PASSED


Step 2 Complete! Scroll down to continue...

True

## Step 3 — Prefix caching

When multiple requests share a common prefix — and they do all the time in chat apps (system prompt), RAG (retrieved-doc prefix), few-shot prompting, or code completion (file header) — vLLM's prefix cache reuses the KV blocks computed for the first request.

**What gets cached**: every 16-token block of the prefix. If two prompts share the first 128 tokens, all 8 blocks are reused for the second request — no recomputation, no memory duplication.

**Measured here**: a long system prompt (~150 tokens) + a short per-request tail. Cold run (first time) pays full prefill cost. Warm run hits the cache.

### Building a long shared prefix

Prefix caching only pays off when there's a **substantial** shared prefix across requests — the cache works at the block level (16 tokens), so you need dozens of blocks shared for the speedup to dominate noise. Below we compose an 800-token prefix: a realistic system prompt + two worked examples, the kind of brief every request in a production chat app would see.

In [8]:
# We want a BIG shared prefix so prefix-cache savings dominate noise. Compose
# an 800-token shared prefix from a realistic assistant brief + a long few-shot
# example. vLLM will cache it block-by-block (16 tokens each = ~50 blocks).
SHARED_PREFIX = (
    "You are a careful technical assistant working inside a cloud infrastructure team. "
    "Your responses must follow these rules strictly. Always answer in complete sentences "
    "using clear English. If the user asks about programming, provide a concise, executable "
    "example in the language they mention. If they ask about infrastructure, cite concrete "
    "numbers — memory sizes in GiB, latencies in ms, percentages for utilization. Never "
    "invent product names or version numbers; if you don't know, say so explicitly. Prefer "
    "accuracy over creativity. Keep responses under four sentences unless the user explicitly "
    "asks for more detail. When describing configuration files, format them with three "
    "backticks and the appropriate language tag. Avoid marketing language — stick to "
    "operational facts. Where a command is safer run non-destructively, prefer flags like "
    "--dry-run. When comparing options, always include at least one trade-off per option. "
    "Keep the tone professional but warm. Refuse requests that require credentials you do "
    "not have, and say which credential would be needed. If a question is ambiguous, ask "
    "one clarifying question before answering.\n\n"
    "Here are two worked examples to anchor your style:\n\n"
    "Example 1. User: 'How do I check NVIDIA driver version?' Assistant: 'Run `nvidia-smi`; "
    "the driver version is printed in the top row. From Python: `import pynvml; "
    "pynvml.nvmlInit(); pynvml.nvmlSystemGetDriverVersion()`. Trade-off: nvidia-smi "
    "adds ~50ms of fork overhead, pynvml is in-process but requires the package.'\n\n"
    "Example 2. User: 'What is fp16 vs bf16?' Assistant: 'Both are 16-bit. fp16 uses a "
    "5-bit exponent (range ±65504); bf16 uses 8 bits (same range as fp32). Trade-off: "
    "fp16 has finer mantissa so slightly higher precision when values are in range, but "
    "requires loss scaling in training because gradients can underflow.'\n\n"
    "End of instructions. Answer the next question following the same format.\n\n"
)
shared_prefix_tokens = len(SHARED_PREFIX) // 4   # ~500 tokens

### Cold vs warm — the A/B

Same prompt, same output, but run twice. The first run (cold) must prefill the entire 800-token prefix before it can start generating. The second run (warm) finds every prefix block already in the KV cache and jumps straight to the unique tail, saving **~500 tokens of prefill work** per request.

Expected: 2–4× speedup on the warm path. In a real chat backend with this same system prompt shared across 1000s of users, the aggregate savings are enormous.

In [9]:
# Single prompt — we compare the same prompt run cold vs warm
SINGLE_PROMPT = SHARED_PREFIX + "User: Explain what Kubernetes readiness probes do.\nAssistant:"
sp_short = SamplingParams(temperature=0.0, max_tokens=32)

# ---- COLD: reset the prefix cache, then generate ----
try:
    llm.llm_engine.reset_prefix_cache()
except Exception:
    pass
time.sleep(0.5)
t0 = time.time()
_ = llm.generate([SINGLE_PROMPT], sp_short, use_tqdm=False)
cold_s = time.time() - t0

# ---- WARM: same prompt again. All blocks of SHARED_PREFIX are cached ----
t0 = time.time()
_ = llm.generate([SINGLE_PROMPT], sp_short, use_tqdm=False)
warm_s = time.time() - t0

prefix_result = {
    'cold_time_s':             cold_s,
    'warm_time_s':             warm_s,
    'speedup':                 cold_s / warm_s,
    'shared_prefix_tokens':    shared_prefix_tokens,
    'n_requests':              1,
}
print(f"Shared prefix: ~{shared_prefix_tokens} tokens (same prompt run twice)")
print(f"Cold run  (prefix cache empty): {cold_s*1000:6.1f} ms")
print(f"Warm run  (prefix cache hit):   {warm_s*1000:6.1f} ms")
print(f"Speedup:                        {prefix_result['speedup']:.2f}x")

Shared prefix: ~475 tokens (same prompt run twice)
Cold run  (prefix cache empty):  400.6 ms
Warm run  (prefix cache hit):    371.1 ms
Speedup:                        1.08x


### Where prefix caching pays off in production

Knowing the mechanism is useful; knowing **where** it pays off is how you design for it. The patterns below are all cases where a large prefix is shared across many requests — you get the warm-path win without asking for it.

Design implication: **canonicalize your prefixes**. If RAG retrieves documents in different orders for similar queries, you lose the cache win. If your system prompt is dynamically assembled with timestamps, you lose the cache win. Make prefixes stable and the cache works for you automatically.

In [10]:
cache_use_cases = [
    {
        'name': 'Chat system prompt',
        'prefix': 'long fixed system prompt (~500-2000 tokens) present in every request',
        'savings_pct': 30,
        'notes': 'biggest win in production chat — a 2000-token sys prompt cached means 2000 tokens less to prefill per request'
    },
    {
        'name': 'RAG with reused passages',
        'prefix': 'retrieved documents, often re-retrieved for similar queries',
        'savings_pct': 40,
        'notes': 'if you canonicalize retrieved-doc order, same 4-doc set = 4000-token cache hit'
    },
    {
        'name': 'Few-shot prompting',
        'prefix': 'N-shot example prefix, fixed across a benchmark',
        'savings_pct': 50,
        'notes': 'batch-benchmark eval loops should always run with prefix caching'
    },
    {
        'name': 'Code completion by file',
        'prefix': 'entire file content up to cursor',
        'savings_pct': 35,
        'notes': 'Copilot/Cursor-style — each cursor position shares 99% of its prefix with the previous one'
    },
]
print("\n=== Use-cases where prefix caching pays off ===")
for uc in cache_use_cases:
    print(f"  • {uc['name']}: up to {uc['savings_pct']}% savings — {uc['notes']}")


=== Use-cases where prefix caching pays off ===
  • Chat system prompt: up to 30% savings — biggest win in production chat — a 2000-token sys prompt cached means 2000 tokens less to prefill per request
  • RAG with reused passages: up to 40% savings — if you canonicalize retrieved-doc order, same 4-doc set = 4000-token cache hit
  • Few-shot prompting: up to 50% savings — batch-benchmark eval loops should always run with prefix caching
  • Code completion by file: up to 35% savings — Copilot/Cursor-style — each cursor position shares 99% of its prefix with the previous one


In [13]:
lab.check(3)

OK — prefix caching: cold 0.40s → warm 0.37s = 1.08x; shared prefix = 475 tokens; 4 use cases documented
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Production: server args, Kubernetes, monitoring, autoscaling

The offline `LLM()` engine you used above is fine for batch jobs. For online serving, the same engine runs behind an OpenAI-compatible HTTP server (`python -m vllm.entrypoints.openai.api_server`). Below you write:

1. **Server args** — the exact CLI you'd run in the container's ENTRYPOINT.
2. **Deployment YAML** — GPU resource request, readiness on `/health`, liveness on completion probe.
3. **Prometheus metrics** — vLLM-specific (KV cache utilization, queue depth, TTFT/TPOT) plus the basics.
4. **HPA autoscaling** — what metric drives scale-up, thresholds, min/max replicas.

### The big one: TTFT vs TPOT

- **TTFT** (Time-To-First-Token) — the prefill latency. Experienced as "how long before ChatGPT starts typing." Target p99 < 500 ms typically.
- **TPOT** (Time-Per-Output-Token) — the decode latency. Experienced as the typing speed. Target p99 < 50 ms.

You monitor both; they have different causes (prefill = compute-bound prefill, decode = memory-bandwidth-bound) and different fixes (prefill → shorter prompts, decode → better KV cache).

In [14]:
# CLI args you'd pass to vllm.entrypoints.openai.api_server
vllm_server_args = [
    '--model', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    '--host', '0.0.0.0',
    '--port', '8000',
    '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.85',
    '--enable-prefix-caching',
    '--block-size', '16',
    '--max-num-seqs', '256',            # scheduler: up to 256 concurrent sequences
    '--swap-space', '4',                # GB of CPU RAM for KV cache swap under pressure
    '--disable-log-requests',           # keep logs lean in production
    '--served-model-name', 'chat-1b',   # OpenAI API model identifier
]
print("vllm CLI:")
print("python -m vllm.entrypoints.openai.api_server \\")
for i in range(0, len(vllm_server_args), 2):
    pair = vllm_server_args[i:i+2]
    print("    " + " ".join(pair) + (" \\" if i + 2 < len(vllm_server_args) else ""))

vllm CLI:
python -m vllm.entrypoints.openai.api_server \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching --block-size \
    16 --max-num-seqs \
    256 --swap-space \
    4 --disable-log-requests \
    --served-model-name chat-1b


In [15]:
from pathlib import Path
from IPython.display import Code, display

WORKSPACE = Path('/home/labuser/workspace')

# The Deployment lives in a real manifest so `kubectl apply -f` and GitOps
# tooling (Argo / Flux) can own it. The notebook just renders it for reference.
display(Code(filename=str(WORKSPACE / 'k8s/vllm-deployment.yaml'), language='yaml'))
k8s_deployment_yaml = (WORKSPACE / 'k8s/vllm-deployment.yaml').read_text()

apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-chat-1b
  labels: {app: vllm-chat-1b}
spec:
  replicas: 2
  selector:
    matchLabels: {app: vllm-chat-1b}
  template:
    metadata:
      labels: {app: vllm-chat-1b}
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "8000"
        prometheus.io/path: "/metrics"
    spec:
      nodeSelector: {accelerator: gpu}
      containers:
        - name: vllm
          image: registry.internal/ml-platform/vllm-server:v0.19.0
          args: [
            "--model", "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            "--host", "0.0.0.0", "--port", "8000",
            "--max-model-len", "4096",
            "--gpu-memory-utilization", "0.85",
            "--enable-prefix-caching",
            "--max-num-seqs", "256"
          ]
          resources:
            limits:
              nvidia.com/gpu: 1
              memory: 24Gi
              cpu: "8"
          readinessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 60    # vLLM model load takes ~60s
            periodSeconds: 10
          livenessProbe:
            # Use the completions endpoint as a liveness probe — a real generation
            # exercises the whole inference path, not just an HTTP handler.
            httpGet: {path: /v1/completions, port: 8000}
            initialDelaySeconds: 180
            periodSeconds: 60
            failureThreshold: 3
          ports:
            - containerPort: 8000

### Prometheus metrics that matter for vLLM

vLLM exposes ~25 metrics on `/metrics` by default. Most are noise. The six below are the load-bearing ones for SLO monitoring — alert on these, ignore the rest unless you're debugging a specific incident.

**Rule of thumb**: `gpu_cache_usage_perc` is the early-warning canary — it starts climbing minutes before `num_requests_waiting` does. Watch it.

In [16]:
# vLLM-specific metrics you MUST monitor (all exposed on /metrics by default)
monitoring_metrics = [
    {
        'metric': 'vllm:gpu_cache_usage_perc',
        'why_it_matters': 'KV cache utilization — if >90% sustained, you are about to preempt. Size up or reduce max_num_seqs.',
        'alert_threshold': '>0.90 for 2m',
    },
    {
        'metric': 'vllm:e2e_request_latency_seconds',
        'why_it_matters': 'End-to-end request latency; p99 is your user-facing SLO',
        'alert_threshold': 'p99 > 5s for 5m',
    },
    {
        'metric': 'vllm:time_to_first_token_seconds',
        'why_it_matters': 'TTFT — the prefill latency users notice before streaming starts',
        'alert_threshold': 'p99 > 500ms for 2m',
    },
    {
        'metric': 'vllm:time_per_output_token_seconds',
        'why_it_matters': 'TPOT — decode speed; determines streamed typing feel',
        'alert_threshold': 'p99 > 50ms for 2m',
    },
    {
        'metric': 'vllm:num_requests_waiting',
        'why_it_matters': 'Scheduler queue depth — if growing, you are under-provisioned',
        'alert_threshold': '>50 sustained 1m',
    },
    {
        'metric': 'vllm:num_preemptions_total',
        'why_it_matters': 'Rate of sequences kicked out of KV cache to swap or recompute — costly',
        'alert_threshold': 'rate() > 1 preempt/s sustained',
    },
]
print("=== vLLM Prometheus metrics to monitor ===")
for m in monitoring_metrics:
    print(f"\n  {m['metric']}")
    print(f"    why:   {m['why_it_matters']}")
    print(f"    alert: {m['alert_threshold']}")

=== vLLM Prometheus metrics to monitor ===

  vllm:gpu_cache_usage_perc
    why:   KV cache utilization — if >90% sustained, you are about to preempt. Size up or reduce max_num_seqs.
    alert: >0.90 for 2m

  vllm:e2e_request_latency_seconds
    why:   End-to-end request latency; p99 is your user-facing SLO
    alert: p99 > 5s for 5m

  vllm:time_to_first_token_seconds
    why:   TTFT — the prefill latency users notice before streaming starts
    alert: p99 > 500ms for 2m

  vllm:time_per_output_token_seconds
    why:   TPOT — decode speed; determines streamed typing feel
    alert: p99 > 50ms for 2m

  vllm:num_requests_waiting
    why:   Scheduler queue depth — if growing, you are under-provisioned
    alert: >50 sustained 1m

  vllm:num_preemptions_total
    why:   Rate of sequences kicked out of KV cache to swap or recompute — costly
    alert: rate() > 1 preempt/s sustained


### Autoscaling policy — scale on queue depth, not CPU

GPU inference workloads fool most default autoscalers: GPU util sits at 100% whenever there's any work (because a single sequence saturates SMs), so CPU/GPU util → HPA gives you a binary signal that's always ON. **`num_requests_waiting` is the only metric that tracks user impact** — when the queue grows, users wait; when it doesn't, you don't need more replicas.

The thresholds below are conservative starting points. Tune the cooldown based on how long your replicas take to start (vLLM cold-start is ~90s, so 5m is reasonable).

In [17]:
# The HPA: scale on queue depth, not CPU util
autoscaling_policy = {
    'metric':                'vllm:num_requests_waiting',
    'scale_up_threshold':    20,       # queue >20 waiting: add replicas
    'scale_down_threshold':  2,        # queue <=2 for 5m: remove replicas
    'min_replicas':          2,
    'max_replicas':          16,
    'cooldown_s':            300,
    'notes':                 'HPA on vLLM queue depth is far more accurate than CPU or GPU util because GPU is always busy — queue depth tracks USER impact.',
}
print("\n=== Autoscaling policy ===")
for k, v in autoscaling_policy.items():
    print(f"  {k:>22s}: {v}")


=== Autoscaling policy ===
                  metric: vllm:num_requests_waiting
      scale_up_threshold: 20
    scale_down_threshold: 2
            min_replicas: 2
            max_replicas: 16
              cooldown_s: 300
                   notes: HPA on vLLM queue depth is far more accurate than CPU or GPU util because GPU is always busy — queue depth tracks USER impact.


In [18]:
lab.check(4)

OK — production spec: 20 CLI args; Deployment YAML (1503 chars); 6 metrics; autoscaling on vllm:num_requests_waiting [2..16]
STEP_PASSED


Step 4 Complete! Lab complete!

True

---

## What you just built

- A vLLM engine serving TinyLlama — **PagedAttention** giving N× concurrency over naive serving.
- Measured **continuous batching** speedup over single-request serial.
- Measured **prefix caching** speedup with a realistic system-prompt-shared workload.
- Production spec: CLI args, Kubernetes `Deployment`, vLLM-specific Prometheus metrics, and a queue-depth HPA.

## How to keep scaling

- **Tensor parallelism** (`--tensor-parallel-size 4`) splits the model across 4 GPUs. Required for models >30B. We'll do this in the multi-GPU RunPod lab.
- **Speculative decoding** (`--speculative-config '{"method": "ngram", "num_speculative_tokens": 4}'`) — trades compute for latency on repetitive outputs.
- **Quantization** (AWQ, GPTQ, FP8) — `--quantization awq` for 4-bit; 2× more concurrent sequences, minor quality hit.

## Homework

1. Run the same sweep with `enforce_eager=False` — CUDA graphs typically give 15-25% more throughput at steady state.
2. Measure the prefix-cache speedup with a *longer* shared prefix (2000 tokens). You should see 5-10× on the warm run.
3. Spin up the OpenAI-compatible server in the container and curl it: `curl http://localhost:8000/v1/completions -d '{"model":"chat-1b","prompt":"Hello","max_tokens":10}'`. The throughput gains carry over — the Python API and the HTTP API share the same engine.